In [77]:
import os
import json
import shutil
import zipfile
import promonet
import requests
import urllib.request
from glob import glob
from tqdm import tqdm

import torch
import torchutil
import torchaudio
from pathlib import Path

## Speech

1. VCTK

In [108]:
# From promonet:
def resample(audio, sample_rate):
    """Resample audio to ProMoNet sample rate"""
    # Cache resampling filter
    key = str(sample_rate)
    if not hasattr(resample, key):
        setattr(
            resample,
            key,
            torchaudio.transforms.Resample(sample_rate, promonet.SAMPLE_RATE))

    # Resample
    return getattr(resample, key)(audio)

In [109]:
str(promonet.CACHE_DIR)

'/home/danzieboy/dev/sensorium-ai/.venv/lib/python3.10/site-packages/promonet-0.0.1-py3.10.egg/data/cache'

In [110]:
# Download zip with progress bar (not as a stream though)

dataset_name = 'vctk'
path = f"./data/{dataset_name}"
if not os.path.exists(path):
    os.makedirs(path)

# cache_dir = f"./cache/{dataset_name}"
cache_dir = f"{str(promonet.CACHE_DIR)}/{dataset_name}"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

url = "https://datashare.ed.ac.uk/download/DS_10283_3443.zip"
filename = url.split('/')[-1]

assert filename.split('.')[-1] == 'zip'

In [111]:
class DownloadProgressBar(tqdm):
  def update_to(self, b=1, bsize=1, tsize=None):
    if tsize is not None:
      self.total = tsize
    self.update(b * bsize - self.n)

In [4]:
# [TODO] Only download if not downloaded/processed yet:

# Download with progress bar
with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=url.split('/')[-1]) as t:
  urllib.request.urlretrieve(url, filename=path, reporthook=t.update_to)
  
# Unzip (with progress bars)
level = 0
while(True):
    # Get zip files (break if no more zip files)
    files = glob(f"{path}/*.zip")
    if len(files) == 0: break
    
    # Unzip
    for file in files:
        with zipfile.ZipFile(file) as zf:
            for member in tqdm(zf.infolist(), desc=f'Extracting {dataset_name}; Level {level+1}; ZipFile "{file.split('/')[-1]}" - '):
                try:
                    zf.extract(member, path)
                except zipfile.error as e:
                    pass
        
        # Delete zip
        os.remove(file)
    level += 1

DS_10283_3443.zip: 11.7GB [7:04:39, 461kB/s]                                 


In [112]:
# File locations
audio_path = f"{path}/wav48_silence_trimmed"
audio_files = sorted(glob(f"{audio_path}/**/*.flac", recursive=True))
# text_path = f"{path}/txt"
# text_files = sorted(glob(f"{text_path}/**/*.txt", recursive=True))

# For some reason, text_files don't align with audio_files
# assert False not in [a[0].split('/')[-2] == a[1].split('/')[-2] for a in list(zip(audio_files, text_files))]

text_files = [
    '/'.join([
        '/'.join(file.split('/')[:-3]), 
        'txt', 
        file.split('/')[-2], 
        ('_'.join(file.split('/')[-1].split('_')[:2]) + '.txt')
    ])
    for file in audio_files
]

# If text file doesn't exist, remove corresponding audio file
audio_files, text_files = list(zip(*[a for a in list(zip(audio_files, text_files)) if os.path.exists(a[1])]))

# Turn strings to paths
# audio_files = [Path(file) for file in audio_files]
# text_files = [Path(file) for file in text_files]

In [115]:
speaker_count = {}
correspondence = {}
pbar = tqdm(list(zip(audio_files, text_files)), desc=f"Formatting {dataset_name}")
for audio_file, text_file in pbar:
    # Get speaker ID
    speaker = audio_file.split('_')[-2]
    if speaker not in speaker_count:
        # (index, count)
        speaker_count[speaker] = [len(speaker_count), 0]
    
    # Update speaker and get current entry
    speaker_count[speaker][1] += 1
    index, count = speaker_count[speaker]
    
    # Load audio
    audio, sample_rate = torchaudio.load(audio_file)
    
    # If audio is too quiet, increase the volume
    maximum = torch.abs(audio).max()
    if maximum < .35:
        audio *= .35 / maximum
        
    # Save at original sampling rate
    speaker_dir = f"{cache_dir}/{index:04d}"
    if not os.path.exists(speaker_dir):
        os.makedirs(speaker_dir)
    output_file = f"{count:06d}.wav"
    torchaudio.save(
        f"{speaker_dir}/{output_file}", 
        audio, 
        sample_rate
    )
    
    # Save Text
    shutil.copyfile(
        text_file,
        f"{speaker_dir}/{output_file.split('.')[0]}.txt"
    )
    
    # Save at system sample rate
    audio = resample(audio, sample_rate)
    torchaudio.save(
        f"{speaker_dir}/{output_file.split('.')[0]}-100.wav",
        audio,
        promonet.SAMPLE_RATE
    )
    
    # Save file stem correspondence
    correspondence[f"{index:04d}/{output_file.split('.')[0]}"] = audio_file.split('/')[-1].split('.')[0]
with open(f'{cache_dir}/correspondence.json', 'w') as correspondence_file:
    json.dump(correspondence, correspondence_file)

Formatting vctk: 100%|██████████| 88156/88156 [27:44<00:00, 52.97it/s]


'/home/danzieboy/dev/sensorium-ai/.venv/lib/python3.10/site-packages/promonet-0.0.1-py3.10.egg/data/cache/vctk/0398/000163.txt'

In [107]:
speaker_dir

'/home/danzieboy/dev/sensorium-ai/.venv/lib/python3.10/site-packages/promonet-0.0.1-py3.10.egg/data/cache/vctk/0398'